In [ ]:
import pandas as pd
import re
import random
import os
from PIL import Image, ImageDraw, ImageFont
import cv2
import sys, os
import json
sys.path.append('../utils')
import text_utils
import image_utils
import numpy as np
from transformers import VisionEncoderDecoderModel
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import TrOCRProcessor
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator


In [ ]:

salish_words, english_words = text_utils.load_wordlist("../sources/roots.csv", "../sources/english.txt")
for w in salish_words[:10]:
    print(repr(w))
def make_text_sample(salish_words, english_words):
    
    salish_sample = " ".join(random.sample(salish_words, k=random.randint(3,5)))
    english_sample = " ".join(random.sample(english_words, k=random.randint(3,5)))
    layouts = [
        salish_sample + random.choice(["."]),
        english_sample + random.choice(["."]),
        salish_sample + random.choice([".", "?"])  + "\n" + english_sample + random.choice([".", "?", "!", "…"])
    ]
    return random.choice(layouts)
os.makedirs("../new_synthetic/images", exist_ok=True)
font_cfg = json.load(open("../sources/fonts_config.json"))
all_fonts = font_cfg["Charis"] +font_cfg["Doulos"] +  font_cfg["NotoSans"]
def random_font():
    fpath = random.choice(all_fonts)
    size = random.randint(24, 64)
    return ImageFont.truetype(fpath, size=size)

def random_layout(img_w, img_h):
    """return margin and line spacing pattern"""
    margin = random.randint(20, 100)
    spacing = random.randint(10, 40)
    return margin, spacing

def render_text_block(text, font, img_w, img_h):
    img = Image.new("RGB", (img_w, img_h), color=255)  # grayscale
    draw = ImageDraw.Draw(img)
    margin, spacing = random_layout(img_w, img_h)
    y = margin
    for line in text.split("\n"):
        draw.text((margin, y), line, font=font, fill=0)  # black text
        bbox = draw.textbbox((0, 0), line, font=font)
        line_height = bbox[3] - bbox[1]
        y += line_height + spacing
    return img

# later:


entries = []
for i in range(5000):
    text = make_text_sample(salish_words, english_words)
    font = random_font()
    img = render_text_block(text, font, img_w=1400, img_h=400)
    fpath = f"../new_synthetic/images/sample_{i}.png"
    img.save(fpath)
    entries.append({'file_name': f"/new_synthetic/images/sample_{i}.png", "text": text})
df = pd.DataFrame(entries)
df.head()

In [ ]:


train_df, test_df = train_test_split(df, test_size=0.2)
# we reset the indices to start from zero
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [ ]:


class Salish(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text 
        file_name = self.df['file_name'][idx]
        text = self.df['text'][idx]
        # prepare image (i.e. resize + normalize)
        image = Image.open(self.root_dir + file_name).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        # add labels (input_ids) by encoding the text
        labels = self.processor.tokenizer(text, 
                                          padding="max_length", 
                                          max_length=self.max_target_length).input_ids
        # important: make sure that PAD tokens are ignored by the loss function
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding

In [ ]:

model_name = "'microsoft/trocr-base-printed'"
processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)




In [ ]:
ipa_extras = "äáä́éɛɛ́íιóúəɔụʕʔx̣šǰčɬ∤ɫʀᴇc̕l̕m̕n̕p̕q̕r̕ṛʀ̕t̕w̕y̕wertyuiopkjhgfdsazxcvbnmʷ"

extra_tokens = list(ipa_extras)
num_added = processor.tokenizer.add_tokens(extra_tokens)
print(f"✅ Added {num_added} new tokens.")

In [ ]:
model.resize_token_embeddings(len(processor.tokenizer))

In [ ]:
train_dataset = Salish(root_dir='../',
                           df=train_df,
                           processor=processor)
eval_dataset = Salish(root_dir='../',
                           df=test_df,
                           processor=processor)

In [ ]:
encoding = train_dataset[0]
for k,v in encoding.items():
  print(k, v.shape)

In [ ]:
image = Image.open(train_dataset.root_dir + train_df['file_name'][0]).convert("RGB")
image

In [ ]:
labels = encoding['labels']
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.decode(labels, skip_special_tokens=True)
print(label_str)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
data_collator = default_data_collator
training_args = Seq2SeqTrainingArguments(
    output_dir="./salish-trocr",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    evaluation_strategy="steps",
    num_train_epochs=5,
    logging_steps=100,
    save_steps=500,
    eval_steps=500,
    learning_rate=5e-5,
    save_total_limit=2,
    remove_unused_columns=False,  # Important for image inputs
)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 (ignored labels) with pad token id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)
